# Week 3: a CNN from scratch for satellite land-use recognition

**Dataset:** EuroSAT RGB, 27,000 Sentinel-2 tiles (64x64) in 10 land-use classes.
**Model:** `SatNet`, 4 conv stages with BatchNorm and global average pooling.

In this notebook:
1. load the data and make a leak-free stratified split
2. compute normalisation statistics from the training split only
3. check the network's shapes, parameter count and receptive field
4. sanity checks: initial loss, gradient norms, overfitting one batch
5. a quick learning-rate sweep, then the full training run
6. evaluation: per-class scores, confusion matrix, confident mistakes, scene-level groups
7. a BatchNorm ablation and predicting a single tile

> In Colab: **Runtime > Change runtime type > T4 GPU**, then run all cells.

In [ ]:
# Setup: works in Google Colab and locally (from the repo root or from this folder)
import os, sys, subprocess

REPO_URL = "https://github.com/abdulloh006/satellite-scene-recognition.git"  # <- your repo
REPO_DIR = "satellite-scene-recognition"

if "google.colab" in sys.modules:
    if not os.path.exists(REPO_DIR) and not os.getcwd().endswith(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    if os.path.exists(REPO_DIR):
        os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()).startswith("week"):
    os.chdir("..")  # opened from week3_cnn/ or week4_transfer/: go to the repo root

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import time
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image

from scenelab import data as D, engine as E, metrics as M, models as MD, plots as P

SEED = 42
E.seed_everything(SEED)
device = E.get_device()

SMOKE = os.environ.get("SMOKE_TEST") == "1"  # tiny run, only used to test the pipeline
GPU = device.type == "cuda"
EPOCHS = 2 if SMOKE else (15 if GPU else 4)
BATCH = 128
WORKERS = 0 if SMOKE else 2

print("PyTorch", torch.__version__, "| device:", device, "| epochs:", EPOCHS)

## 1. Data: load EuroSAT and split 70 / 15 / 15 per class

In [ ]:
base = D.load_eurosat("./data")            # PIL images, ~90 MB download the first time
CLASSES = D.CLASSES
targets = np.asarray(base.targets)

train_idx, val_idx, test_idx = D.stratified_split(targets, (0.70, 0.15, 0.15), seed=SEED)
print(f"total {len(base):,} | train {len(train_idx):,} | val {len(val_idx):,} | test {len(test_idx):,}")
print("train images per class:", D.class_counts(train_idx, targets))

In [ ]:
rng = np.random.default_rng(SEED)
one_per_class = [int(rng.choice(np.flatnonzero(targets == c))) for c in range(10) for _ in range(2)]
P.show_tiles(base, one_per_class, CLASSES, cols=10, title="Two random tiles per class (64x64, 10 m/pixel)")
P.plot_class_counts(np.bincount(targets, minlength=10), CLASSES,
                    "EuroSAT is mildly imbalanced (2,000-3,000 per class)")

## 2. Normalisation statistics from the **training split only**

Computing mean/std on all data would leak information from the test set into preprocessing.

In [ ]:
MEAN, STD = D.compute_mean_std(base, train_idx, max_images=500 if SMOKE else 4000, seed=SEED)
print("EuroSAT RGB mean:", MEAN)
print("EuroSAT RGB std :", STD)
print("ImageNet mean   :", D.IMAGENET_MEAN, "<- compare: how different are satellite colours?")

train_tf = D.make_transforms(MEAN, STD, augment=True)
eval_tf = D.make_transforms(MEAN, STD)

train_loader = D.make_loader(base, train_idx, train_tf, BATCH, shuffle=True, num_workers=WORKERS)
val_loader = D.make_loader(base, val_idx, eval_tf, 256, num_workers=WORKERS)
test_loader = D.make_loader(base, test_idx, eval_tf, 256, num_workers=WORKERS)

x, y = next(iter(train_loader))
print("batch:", tuple(x.shape), "| labels:", tuple(y.shape),
      f"| pixel range after normalisation {x.min():.2f} .. {x.max():.2f}")

## 3. The model: shapes, parameters, receptive field

In [ ]:
model = MD.SatNet(num_classes=10).to(device)
print(model)

# Trace one fake tile through the network
h = torch.randn(1, 3, 64, 64, device=device)
model.eval()
with torch.no_grad():
    for name, layer in model.features.named_children():
        h = layer(h)
        if isinstance(layer, (nn.Conv2d, nn.MaxPool2d)):
            print(f"{name:>2} {layer.__class__.__name__:<10} -> {tuple(h.shape)}")
    for layer in model.head:
        h = layer(h)
        print(f"   {layer.__class__.__name__:<17} -> {tuple(h.shape)}")

In [ ]:
mlp = nn.Sequential(nn.Flatten(), nn.Linear(3 * 64 * 64, 256), nn.ReLU(), nn.Linear(256, 10))
print(f"SatNet parameters   : {MD.count_params(model):>10,}")
print(f"Plain MLP parameters: {MD.count_params(mlp):>10,}  <- more weights, and no notion of 'nearby pixels'\n")

for name, p in model.named_parameters():
    print(f"{name:<24}{str(tuple(p.shape)):<20}{p.numel():>9,}")

In [ ]:
print("Receptive field (in input pixels) after each layer:")
for name, r, jump in MD.receptive_fields(model.features):
    print(f"  {name:<14} sees {r:>3} x {r:<3} pixels  (stride so far {jump})")
print("\nAfter global average pooling the classifier combines features from the whole 64x64 tile.")

## 4. Sanity checks before training

* With random weights, the loss should be close to ln(10) = 2.303 (a uniform guess over 10 classes).
* Every layer should receive a non-zero gradient.
* A healthy model + training loop should be able to **memorise a single batch**. If it cannot,
  there is a bug, and there is no point training on the full data.

In [ ]:
criterion = nn.CrossEntropyLoss()
model.train()
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)
model.zero_grad()
loss = criterion(model(x), y)
loss.backward()
print(f"initial loss {loss.item():.3f}  vs  ln(10) = {math.log(10):.3f}")
for name, p in model.named_parameters():
    if p.grad is not None and p.dim() > 1:
        print(f"  {name:<22} grad norm {p.grad.norm().item():.4f}")

In [ ]:
E.seed_everything(SEED)
probe = MD.SatNet(dropout=0.0).to(device)
opt = torch.optim.Adam(probe.parameters(), lr=1e-3)
xb, yb = x[:32], y[:32]
steps = 30 if SMOKE else 150
for step in range(1, steps + 1):
    probe.train()
    opt.zero_grad()
    l = criterion(probe(xb), yb)
    l.backward()
    opt.step()
    if step % (steps // 5) == 0:
        print(f"step {step:>3}: loss on the same 32 images = {l.item():.4f}")
print("Loss should approach 0: the model can memorise, so capacity and loop are fine.")

## 5. Quick learning-rate sweep

Two epochs on a 3,000-image subset for each learning rate, always from the same initial weights.
*My prediction before running:* ________ (write it down, then check).

In [ ]:
sub_idx = D.per_class_subset(train_idx, targets, 30 if SMOKE else 300, SEED)
sub_loader = D.make_loader(base, sub_idx, train_tf, BATCH, shuffle=True, num_workers=WORKERS)

def quick_lr_test(lr, epochs=2):
    E.seed_everything(SEED)
    m = MD.SatNet().to(device)
    o = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    for _ in range(epochs):
        tl, _ = E.run_epoch(m, sub_loader, device, criterion, o)
    _, va = E.run_epoch(m, val_loader, device, criterion)
    return tl, va

sweep = {}
for lr in [1e-2, 3e-3, 1e-3, 1e-4]:
    sweep[lr] = quick_lr_test(lr)
    print(f"lr {lr:<7} last train loss {sweep[lr][0]:.3f} | val acc {sweep[lr][1]:.1%}")

BEST_LR = max(sweep, key=lambda k: sweep[k][1])
print("\nbest lr on this sweep:", BEST_LR)

## 6. Full training run

AdamW + OneCycle schedule (warm up, then anneal). `fit` keeps the weights from the
epoch with the best validation accuracy, not simply the last epoch.

In [ ]:
E.seed_everything(SEED)
model = MD.SatNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=BEST_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=BEST_LR, epochs=EPOCHS, steps_per_epoch=len(train_loader))

history = E.fit(model, train_loader, val_loader, device, EPOCHS, optimizer,
                criterion, scheduler, tag="SatNet")

os.makedirs("checkpoints", exist_ok=True)
torch.save({"state_dict": model.state_dict(), "mean": MEAN, "std": STD, "classes": CLASSES},
           "checkpoints/satnet_week3.pt")
print("saved checkpoints/satnet_week3.pt")

In [ ]:
P.plot_histories({"SatNet (scratch, full data)": history})
plt.figure(figsize=(5, 2.5)); plt.plot(range(1, EPOCHS + 1), history["lr"], "o-")
plt.title("learning rate at end of each epoch (OneCycle)"); plt.xlabel("epoch"); plt.tight_layout(); plt.show()

## 7. Evaluation on the held-out test set

In [ ]:
probs, ys = E.predict(model, test_loader, device)
preds = probs.argmax(1)
print(f"Test accuracy: {(preds == ys).float().mean():.2%}\n")
M.per_class_table(ys, preds, CLASSES)

cm = M.confusion(ys, preds, 10)
P.plot_confusion(cm, CLASSES, title="SatNet test set")
print("Three most common mistakes:")
for t, p, n in M.top_confusions(cm, CLASSES):
    print(f"  {t} predicted as {p}: {n} times")

In [ ]:
print("The model's most confident mistakes:")
P.show_confident_mistakes(base, test_idx, probs, ys, CLASSES, k=8)

### Object-level vs scene-level accuracy

Classes are grouped into **urban / agriculture / natural / water**. Confusing Pasture with
AnnualCrop is a small error; calling a River a Highway is a big one.

In [ ]:
g_acc, g_cm = M.group_accuracy(ys, preds, CLASSES, D.GROUPS)
print(f"10-class accuracy          : {(preds == ys).float().mean():.2%}")
print(f"4-group (scene) accuracy   : {g_acc:.2%}")
P.plot_confusion(g_cm, list(D.GROUPS), title="Group-level confusion")

## 8. Does the output-size formula match PyTorch?

In [ ]:
probe_in = torch.randn(1, 3, 64, 64)
for k, p, s, d in [(3, 0, 1, 1), (3, 1, 1, 1), (3, 1, 2, 1), (5, 2, 2, 1), (3, 2, 1, 2)]:
    real = nn.Conv2d(3, 4, k, stride=s, padding=p, dilation=d)(probe_in).shape[-1]
    print(f"conv k={k} p={p} s={s} d={d}: formula {MD.conv_out_size(64, k, p, s, d):>2} | PyTorch {real}")
real = nn.MaxPool2d(2)(probe_in).shape[-1]
print(f"maxpool k=2 s=2      : formula {MD.conv_out_size(64, 2, 0, 2):>2} | PyTorch {real}")

## 9. Ablation: what does BatchNorm buy us?

Same subset, same learning rate, same number of epochs; the only difference is BatchNorm.

In [ ]:
def short_run(batchnorm, epochs=2 if SMOKE else 5):
    E.seed_everything(SEED)
    m = MD.SatNet(batchnorm=batchnorm).to(device)
    o = torch.optim.AdamW(m.parameters(), lr=BEST_LR, weight_decay=1e-4)
    return E.fit(m, sub_loader, val_loader, device, epochs, o, criterion,
                 tag=f"BN={batchnorm}", restore_best=False)

h_bn, h_nobn = short_run(True), short_run(False)
P.plot_histories({"with BatchNorm": h_bn, "without BatchNorm": h_nobn})

## 10. Predict a single tile

In [ ]:
def predict_tile(path=None, topk=3):
    """Classify a tile from disk, or a random test tile if no path is given.

    EuroSAT tiles are 64x64 at 10 m/pixel, so your own image should be an aerial or satellite view
    (e.g. a Google Maps satellite screenshot zoomed out to show roughly 640 m x 640 m).
    """
    if path is None:
        i = int(np.random.default_rng().choice(test_idx))
        img, true = base[i]
        title = f"test tile, true: {CLASSES[true]}"
    else:
        img = Image.open(path).convert("RGB")
        s = min(img.size)
        img = img.crop(((img.width - s) // 2, (img.height - s) // 2,
                        (img.width + s) // 2, (img.height + s) // 2)).resize((64, 64), Image.LANCZOS)
        title = os.path.basename(path)
    x = eval_tf(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        p = model(x).softmax(1)[0].cpu()
    plt.figure(figsize=(3, 3)); plt.imshow(img.resize((192, 192), Image.NEAREST)); plt.axis("off")
    plt.title(title, fontsize=9); plt.show()
    for score, c in zip(*p.topk(topk)):
        print(f"  {CLASSES[c]:<22}{score:.1%}")

predict_tile()                              # random test tile
predict_tile("samples/eurosat_river.png")   # sample tile; try your own satellite screenshot too

## Observations (write these yourself after running)

* Test accuracy, and how it compares with the HOG + SVM baseline from Week 2:
* Which classes are hardest, and why do you think so (look at the confident mistakes)?
* Was your learning-rate prediction right?
* What did BatchNorm change in the learning curves?
* Is there a train/validation gap? Is the model over- or under-fitting?